# GeoDataFrame Visualization of Cell-Type Annotations

In [ ]:
import spatialdata as sd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
import pandas as pd
from shapely.geometry import Polygon
from Xenium_5K_IPF_constants import SAMPLES_SYMBOLS, MAJOR_CLUSTER_COLORS, SUBTYPE_HATCHES, WORK_PATH

# Option
plt.rcParams["font.family"] = "Fira Sans" 
%matplotlib ipympl

In [ ]:
celltype_df=pd.read_csv(f"{WORK_PATH}CSV/013_merge_type.csv", index_col="cell_id")
subtype_colors = {sb:MAJOR_CLUSTER_COLORS[celltype_df.loc[celltype_df["subtype"]==sb, "cell_type_major"].iloc[0]] for sb in list(SUBTYPE_HATCHES)}

In [ ]:
for n, ss in enumerate(SAMPLES_SYMBOLS):
    sp_adata = sd.SpatialData.read(f"{WORK_PATH}SpatialData/{ss}")
    gdf = sp_adata.shapes["cell_boundaries"]
    del sp_adata
    
    df_c = celltype_df[celltype_df.index.str.endswith(str(n))]
    df_c.index = [m[:-1]+"1" for m in df_c.index]
    
    gdf["subtype"] = gdf.index.map(df_c["subtype"].to_dict())
    
    with plt.ioff():
        fig, ax = plt.subplots(figsize=(32, 28))
        fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
        fig.set_facecolor("none")
        ax.axis("off")
        
        for sb, sb_df in gdf.groupby("subtype"):
            sb_df.plot(ax=ax, 
                       facecolor=subtype_colors[sb], 
                      )
                
            
        ax.hlines(y=ax.get_ylim()[1]+10, xmin=ax.get_xlim()[1]-1010, xmax=ax.get_xlim()[1]-10, color='black', lw=fig.get_figheight())
        ax.set_ylim(ax.get_ylim()[1], ax.get_ylim()[0])
        fig.savefig(f"{WORK_PATH}Figure/014_major_type_{ss}_gdf_alpha.png", dpi=1000, format="png", bbox_inches=None)


In [ ]:
for n, ss in enumerate(SAMPLES_SYMBOLS):
    sp_adata = sd.SpatialData.read(f"{WORK_PATH}SpatialData/{ss}")
    gdf = sp_adata.shapes["cell_boundaries"]
    del sp_adata
    
    df_c = celltype_df[celltype_df.index.str.endswith(str(n))]
    df_c.index = [m[:-1]+"1" for m in df_c.index]
    
    gdf["subtype"] = gdf.index.map(df_c["subtype"].to_dict())
    
    with plt.ioff():
        fig, ax = plt.subplots(figsize=(160, 140))
        fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
        fig.set_facecolor("white")
        ax.set_facecolor("white")
        
        for sb, sb_df in gdf.groupby("subtype"):
            sb_df.plot(ax=ax, 
                       facecolor=subtype_colors[sb], 
                       hatch=SUBTYPE_HATCHES[sb]
                      )
                
            
        ax.hlines(y=ax.get_ylim()[1]+10, xmin=ax.get_xlim()[1]-1010, xmax=ax.get_xlim()[1]-10, color='black', lw=fig.get_figheight())
        ax.set_ylim(ax.get_ylim()[1], ax.get_ylim()[0])
        fig.savefig(f"{WORK_PATH}Figure/014_all_subtype_{ss}_gdf.png", dpi=200, format="png", bbox_inches=None)


In [ ]:
legend_cls_5 = [
    'Epithelial',
    'B/Plasma cell',
    'Fibroblast',
    'NK/T cell',
    'Monocyte/MΦ',
    'SMC/Pericyte',
    'Endothelial',
    'Dendritic cell',    
    "Granulocyte",
    'FRC/FDC',
]

In [ ]:
# legend
sub_dict={}
for mc in legend_cls_5:
    sub_list = celltype_df.loc[celltype_df["cell_type_major"]==mc, "subtype"].unique()
    sub_list = [sb for sb in list(SUBTYPE_HATCHES) if sb in sub_list]
    sub_dict[mc] = sub_list


In [ ]:
legend_col = {cls: MAJOR_CLUSTER_COLORS[cls] for cls in legend_cls_5}

In [ ]:
# ncols = 5
fig = plt.figure(figsize=(18, 5.7))
fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
fig.set_facecolor("white")

y_start = 0
title_space = 0.13
sq_mk = [0.2, 0.076]
h_space = 1.8

for n, mc in enumerate(legend_cls_5):
    y_wd = len(sub_dict[mc])
    y_start += y_wd
    if n < 2:
        ax = fig.add_axes([0.02, 1-title_space-y_start*sq_mk[1], sq_mk[0], y_wd*sq_mk[1]])
    elif n < 4:
        ax = fig.add_axes([0.2, 1-title_space-y_start*sq_mk[1], sq_mk[0], y_wd*sq_mk[1]])
    elif n < 6:
        ax = fig.add_axes([0.42, 1-title_space-y_start*sq_mk[1], sq_mk[0], y_wd*sq_mk[1]])
    elif n < 8:
        ax = fig.add_axes([0.66, 1-title_space-y_start*sq_mk[1], sq_mk[0], y_wd*sq_mk[1]])
    else:
        ax = fig.add_axes([0.85, 1-title_space-y_start*sq_mk[1], sq_mk[0], y_wd*sq_mk[1]])

    if n % 2 != 0:
        y_start = 0
    else:
        y_start += h_space
        
    fig.text(s=f"    {mc}    ", x=ax.get_position().x0, y=ax.get_position().y1+0.01, color="black", weight=600, ha="left", va="bottom", size=25,
             bbox=dict(facecolor=legend_col[mc], edgecolor='#3E3E3E', pad=0.12, boxstyle='round')
             )

    ax.set_ylim(-0.05,1)
    ax.set_xlim(0,1)
    ax.set_yticks([0,1])
    ax.set_facecolor("none")
    ax.set_ylabel("")
    ax.axis("off")
        
    for m,sb in enumerate(sub_dict[mc]):

        ax.add_patch(matplotlib.patches.Rectangle(
            xy=(0, 1-(m+1)/y_wd), 
            width=0.17, 
            height=0.77/y_wd, 
            edgecolor='#3E3E3E',
            facecolor=legend_col[mc], 
            hatch=SUBTYPE_HATCHES[sb])
        )
            
        ax.text(s=sb, x=0.22, y=1-(m+0.6)/y_wd, color="black", ha="left", va="center", size=23)

fig.savefig(f"{WORK_PATH}Figure/014_all_subtype_legend_ncol5.png", dpi=300, format="png", bbox_inches=None)

In [ ]:
legend_cls_2 = [
    'Epithelial',
    'Fibroblast',
    'Endothelial',
    'Monocyte/MΦ',
    'SMC/Pericyte',
    'NK/T cell',
    'B/Plasma cell',
    'Dendritic cell',
    'Granulocyte',
    'FRC/FDC']

In [ ]:
# ncols = 2
fig = plt.figure(figsize=(7, 11))
fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
fig.set_facecolor("white")

y_start = 0
title_space = 0.05
sq_mk = [0.45, 0.034]
h_space = 1.9
for n, mc in enumerate(legend_cls_2):
    y_wd = len(sub_dict[mc])
    y_start += y_wd
    if n < 4:
        ax = fig.add_axes([0.02, 1-title_space-y_start*sq_mk[1], sq_mk[0], y_wd*sq_mk[1]])
    else:
        ax = fig.add_axes([0.55, 1-title_space-y_start*sq_mk[1], sq_mk[0], y_wd*sq_mk[1]])
        

    if n == 3:
        y_start = 0
    else:
        y_start += h_space
        
    fig.text(s=f"    {mc}    ", x=ax.get_position().x0, y=ax.get_position().y1+0.01, color="black", weight=600, ha="left", va="bottom", size=21,
             bbox=dict(facecolor=legend_col[mc], edgecolor='#3E3E3E', pad=0.12, boxstyle='round'))


    ax.set_ylim(-0.05,1)
    ax.set_xlim(0,1)
    ax.set_yticks([0,1])
    ax.set_facecolor("none")
    ax.set_ylabel("")
    ax.axis("off")
        
    for m,sb in enumerate(sub_dict[mc]):

        ax.add_patch(matplotlib.patches.Rectangle(
            xy=(0, 1-(m+1)/y_wd), 
            width=0.17, 
            height=0.77/y_wd, 
            ec="#3E3E3E", 
            facecolor=legend_col[mc], 
            hatch=SUBTYPE_HATCHES[sb])
        )
            
        ax.text(s=sb, x=0.22, y=1-(m+0.6)/y_wd, color="black", ha="left", va="center", size=19)

fig.savefig(f"{pathWORK_PATH}Figure/014_all_subtype_legend_ncol2.png", dpi=300, format="png", bbox_inches=None)

In [ ]:
legend_cls_6 = [
    'Epithelial',
    'Fibroblast', 'B/Plasma cell',
    'Monocyte/MΦ', "Granulocyte",
    'Endothelial', 'FRC/FDC',
    'NK/T cell', 'SMC/Pericyte',
    'Dendritic cell',    
]

In [ ]:
# ncols = 7
fig = plt.figure(figsize=(22, 4.8))
fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
fig.set_facecolor("white")

y_start = 0
title_space = 0.13
sq_mk = [0.2, 0.095]
h_space = 1.6

for n, mc in enumerate(legend_cls_6):
    y_wd = len(sub_dict[mc])
    y_start += y_wd
    if n < 1:
        ax = fig.add_axes([0.005, 1-title_space-y_start*sq_mk[1], sq_mk[0], y_wd*sq_mk[1]])
    elif n < 3:
        ax = fig.add_axes([0.15, 1-title_space-y_start*sq_mk[1], sq_mk[0], y_wd*sq_mk[1]])
    elif n < 5:
        ax = fig.add_axes([0.35, 1-title_space-y_start*sq_mk[1], sq_mk[0], y_wd*sq_mk[1]])
    elif n < 7:
        ax = fig.add_axes([0.56, 1-title_space-y_start*sq_mk[1], sq_mk[0], y_wd*sq_mk[1]])
    elif n < 9:
        ax = fig.add_axes([0.73, 1-title_space-y_start*sq_mk[1], sq_mk[0], y_wd*sq_mk[1]])
    else:
        ax = fig.add_axes([0.86, 1-title_space-y_start*sq_mk[1], sq_mk[0], y_wd*sq_mk[1]])

    if n % 2 == 0:
        y_start = 0
    else:
        y_start += h_space
        
    fig.text(s=f"    {mc}    ", x=ax.get_position().x0, y=ax.get_position().y1+0.01, color="black", weight=600, ha="left", va="bottom", size=25,
             bbox=dict(facecolor=legend_col[mc], edgecolor='#3E3E3E', pad=0.12, boxstyle='round')
             )

    ax.set_ylim(-0.05,1)
    ax.set_xlim(0,1)
    ax.set_yticks([0,1])
    ax.set_facecolor("none")
    ax.set_ylabel("")
    ax.axis("off")
        
    for m,sb in enumerate(sub_dict[mc]):

        ax.add_patch(matplotlib.patches.Rectangle(
            xy=(0, 1-(m+1)/y_wd), 
            width=0.17, 
            height=0.77/y_wd, 
            edgecolor='#3E3E3E',
            facecolor=legend_col[mc], 
            hatch=SUBTYPE_HATCHES[sb])
        )
            
        ax.text(s=sb, x=0.22, y=1-(m+0.6)/y_wd, color="black", ha="left", va="center", size=23)

fig.savefig(f"{WORK_PATH}Figure/014_all_subtype_legend_ncol6.png", dpi=300, format="png", bbox_inches=None)